# Silver Layer: Cleaning & Validation

Build cleaned silver tables from `dev.bronze` with primary-key uniqueness, foreign-key integrity, and quarantine routing.
Each table follows the pattern: **read bronze → validate (NULL_PK, DUPLICATE_RECORD, INVALID_FOREIGN_KEY) → split clean vs quarantine → write to dev.silver**.

**Processing order** (respecting FK dependencies):
1. region (no FK)
2. nation (FK → region)
3. supplier (FK → nation)
4. customer (FK → nation)
5. part (no FK)
6. partsupp (FK → part, supplier)
7. orders (FK → customer)
8. lineitem (FK → orders, part, supplier)

In [0]:
%python
dbutils.widgets.text("catalog_name" ,"dev")
catalog_name = dbutils.widgets.get("catalog_name")

In [0]:
%python
# ──────────────────────────────────────────
# Common imports — run this cell before any silver cell below
# ──────────────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("✅ Imports ready: F (functions), Window, catalog_name =", catalog_name)

In [0]:
%python
# ═══════════════════════════════════════════
# 1. REGION — no foreign keys
#    PK: r_regionkey
# ═══════════════════════════════════════════

# ── 1. Read from bronze with type casting & string cleaning ──
#     • Numeric PK  → cast to LONG, empty string → NULL
#     • String fields → trim + UPPER, empty → NULL (via nullif)
#     • Comment      → trim only (keep original casing)
df = spark.read.table(f'{catalog_name}.bronze.region').select(
    F.when(
        F.col('r_regionkey').cast('string') == '', None
    ).otherwise(F.col('r_regionkey').cast('long')).alias('r_regionkey'),

    F.nullif(F.trim(F.upper(F.col('r_name'))), F.lit('')).alias('r_name'),
    F.trim(F.col('r_comment')).alias('r_comment')
)

# ── 2. Validation ──
#     • DUPLICATE_RECORD : same PK appears more than once
#     • NULL_PK          : primary key is missing
#     • NULL_NAME        : r_name is null or empty after trim
df_checked = (
    df.withColumn(
        'rnk',
        F.row_number().over(
            Window.partitionBy('r_regionkey')
            .orderBy(F.desc('r_name'))
        )
    )
    .withColumn(
        'reason',
        F.concat_ws(
            ', ',
            F.when(F.col('rnk') > 1, F.lit('DUPLICATE_RECORD')),
            F.when(F.col('r_regionkey').isNull(), F.lit('NULL_PK')),
            F.when(F.col('r_name').isNull(), F.lit('NULL_NAME'))
        )
    )
)

# ── 3. Split clean vs quarantine ──
df_cleaned = (
    df_checked
    .filter(F.col('reason') == '')
    .drop('rnk', 'reason')
)

df_quarantine = (
    df_checked
    .filter(F.col('reason') != '')
    .drop('rnk')
)

# ── 4. Display & save ──
display(df_cleaned)
display(df_quarantine)

df_cleaned.write.mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('dev.silver.region')

df_quarantine.write.mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('dev.silver.quarantine_region')

print("✅ region: silver + quarantine tables created")

In [0]:
%python
# ═══════════════════════════════════════════
# 2. NATION — FK: n_regionkey → region.r_regionkey
#    PK: n_nationkey
# ═══════════════════════════════════════════

# ── 1. Read from bronze with type casting & string cleaning ──
df = spark.read.table(f'{catalog_name}.bronze.nation').select(
    F.when(
        F.col('n_nationkey').cast('string') == '', None
    ).otherwise(F.col('n_nationkey').cast('long')).alias('n_nationkey'),

    F.nullif(F.trim(F.upper(F.col('n_name'))), F.lit('')).alias('n_name'),
    F.when(
        F.col('n_regionkey').cast('string') == '', None
    ).otherwise(F.col('n_regionkey').cast('long')).alias('n_regionkey'),
    F.trim(F.col('n_comment')).alias('n_comment')
)

# ── 2. Load FK reference table ──
region_keys = (
    spark.read.table('dev.silver.region')
    .select(F.col('r_regionkey').alias('valid_regionkey'))
    .distinct()
)

# ── 3. Validation ──
df_checked = (
    df.withColumn(
        'rnk',
        F.row_number().over(
            Window.partitionBy('n_nationkey')
            .orderBy(F.desc('n_regionkey'))
        )
    )
    .join(
        region_keys,
        F.col('n_regionkey') == F.col('valid_regionkey'),
        'left'
    )
    .withColumn(
        'reason',
        F.concat_ws(
            ', ',
            F.when(F.col('rnk') > 1, F.lit('DUPLICATE_RECORD')),
            F.when(F.col('n_nationkey').isNull(), F.lit('NULL_PK')),
            F.when(F.col('n_name').isNull(), F.lit('NULL_NAME')),
            F.when(
                F.col('n_regionkey').isNotNull() &
                F.col('valid_regionkey').isNull(),
                F.lit('INVALID_FOREIGN_KEY')
            )
        )
    )
    .drop('valid_regionkey')
)

# ── 4. Split clean vs quarantine ──
df_cleaned = (
    df_checked
    .filter(F.col('reason') == '')
    .drop('rnk', 'reason')
)

df_quarantine = (
    df_checked
    .filter(F.col('reason') != '')
    .drop('rnk')
)

# ── 5. Display & save ──
display(df_cleaned)
display(df_quarantine)

df_cleaned.write.mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('dev.silver.nation')

df_quarantine.write.mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('dev.silver.quarantine_nation')

print("✅ nation: silver + quarantine tables created")

In [0]:
%python
# ═══════════════════════════════════════════
# 3. SUPPLIER — FK: s_nationkey → nation.n_nationkey
#    PK: s_suppkey
# ═══════════════════════════════════════════

# ── 1. Read from bronze with type casting & string cleaning ──
df = spark.read.table(f'{catalog_name}.bronze.supplier').select(
    F.when(
        F.col('s_suppkey').cast('string') == '', None
    ).otherwise(F.col('s_suppkey').cast('long')).alias('s_suppkey'),

    F.nullif(F.trim(F.upper(F.col('s_name'))), F.lit('')).alias('s_name'),
    F.trim(F.col('s_address')).alias('s_address'),
    F.when(
        F.col('s_nationkey').cast('string') == '', None
    ).otherwise(F.col('s_nationkey').cast('long')).alias('s_nationkey'),
    F.trim(F.col('s_phone')).alias('s_phone'),
    F.when(
        F.col('s_acctbal').cast('string') == '', None
    ).otherwise(F.col('s_acctbal').cast('decimal(18,2)')).alias('s_acctbal'),
    F.trim(F.col('s_comment')).alias('s_comment')
)

# ── 2. Load FK reference table ──
nation_keys = (
    spark.read.table('dev.silver.nation')
    .select(F.col('n_nationkey').alias('valid_nationkey'))
    .distinct()
)

# ── 3. Validation ──
#     • INVALID_PHONE            : phone does not end with 10 digits (after removing dashes)
#     • NEGATIVE_ACCOUNT_BALANCE : s_acctbal < 0
df_checked = (
    df.withColumn(
        'rnk',
        F.row_number().over(
            Window.partitionBy('s_suppkey')
            .orderBy(F.desc('s_nationkey'))
        )
    )
    .join(
        nation_keys,
        F.col('s_nationkey') == F.col('valid_nationkey'),
        'left'
    )
    .withColumn(
        'reason',
        F.concat_ws(
            ', ',
            F.when(F.col('rnk') > 1, F.lit('DUPLICATE_RECORD')),
            F.when(F.col('s_suppkey').isNull(), F.lit('NULL_PK')),
            F.when(F.col('s_name').isNull(), F.lit('NULL_NAME')),
            F.when(
                F.col('s_nationkey').isNotNull() &
                F.col('valid_nationkey').isNull(),
                F.lit('INVALID_FOREIGN_KEY')
            ),
            F.when(
                F.col('s_phone').isNull() |
                ~F.regexp_replace(F.col('s_phone'), '-', '').rlike('\\d{10}$'),
                F.lit('INVALID_PHONE')
            ),
            F.when(
                F.col('s_acctbal').isNotNull() & (F.col('s_acctbal') < 0),
                F.lit('NEGATIVE_ACCOUNT_BALANCE')
            )
        )
    )
    .drop('valid_nationkey')
)

# ── 4. Split clean vs quarantine ──
df_cleaned = (
    df_checked
    .filter(F.col('reason') == '')
    .drop('rnk', 'reason')
)

df_quarantine = (
    df_checked
    .filter(F.col('reason') != '')
    .drop('rnk')
)

# ── 5. Display & save ──
display(df_cleaned.limit(20))
display(df_quarantine.limit(20))

df_cleaned.write.mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('dev.silver.supplier')

df_quarantine.write.mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('dev.silver.quarantine_supplier')

print("✅ supplier: silver + quarantine tables created")

In [0]:
%python
# ═══════════════════════════════════════════
# 4. CUSTOMER — FK: c_nationkey → nation.n_nationkey
#    PK: c_custkey
# ═══════════════════════════════════════════

# ── 1. Read from bronze with type casting & string cleaning ──
df = spark.read.table(f'{catalog_name}.bronze.customer').select(
    F.when(
        F.col('c_custkey').cast('string') == '', None
    ).otherwise(F.col('c_custkey').cast('long')).alias('c_custkey'),

    F.nullif(F.trim(F.upper(F.col('c_name'))), F.lit('')).alias('c_name'),
    F.trim(F.col('c_address')).alias('c_address'),
    F.when(
        F.col('c_nationkey').cast('string') == '', None
    ).otherwise(F.col('c_nationkey').cast('long')).alias('c_nationkey'),
    F.trim(F.col('c_phone')).alias('c_phone'),
    F.when(
        F.col('c_acctbal').cast('string') == '', None
    ).otherwise(F.col('c_acctbal').cast('decimal(18,2)')).alias('c_acctbal'),
    F.nullif(F.trim(F.upper(F.col('c_mktsegment'))), F.lit('')).alias('c_mktsegment'),
    F.trim(F.col('c_comment')).alias('c_comment')
)

# ── 2. Load FK reference table ──
nation_keys = (
    spark.read.table('dev.silver.nation')
    .select(F.col('n_nationkey').alias('valid_nationkey'))
    .distinct()
)

# ── 3. Validation ──
#     • INVALID_PHONE            : phone does not end with 10 digits
#     • NEGATIVE_ACCOUNT_BALANCE : c_acctbal < 0
#     • NULL_MKTSEGMENT          : c_mktsegment is null or empty
df_checked = (
    df.withColumn(
        'rnk',
        F.row_number().over(
            Window.partitionBy('c_custkey')
            .orderBy(F.desc('c_nationkey'))
        )
    )
    .join(
        nation_keys,
        F.col('c_nationkey') == F.col('valid_nationkey'),
        'left'
    )
    .withColumn(
        'reason',
        F.concat_ws(
            ', ',
            F.when(F.col('rnk') > 1, F.lit('DUPLICATE_RECORD')),
            F.when(F.col('c_custkey').isNull(), F.lit('NULL_PK')),
            F.when(F.col('c_name').isNull(), F.lit('NULL_NAME')),
            F.when(
                F.col('c_nationkey').isNotNull() &
                F.col('valid_nationkey').isNull(),
                F.lit('INVALID_FOREIGN_KEY')
            ),
            F.when(
                F.col('c_phone').isNull() |
                ~F.regexp_replace(F.col('c_phone'), '-', '').rlike('\\d{10}$'),
                F.lit('INVALID_PHONE')
            ),
            F.when(
                F.col('c_acctbal').isNotNull() & (F.col('c_acctbal') < 0),
                F.lit('NEGATIVE_ACCOUNT_BALANCE')
            ),
            F.when(F.col('c_mktsegment').isNull(), F.lit('NULL_MKTSEGMENT'))
        )
    )
    .drop('valid_nationkey')
)

# ── 4. Split clean vs quarantine ──
df_cleaned = (
    df_checked
    .filter(F.col('reason') == '')
    .drop('rnk', 'reason')
)

df_quarantine = (
    df_checked
    .filter(F.col('reason') != '')
    .drop('rnk')
)

# ── 5. Display & save ──
display(df_cleaned.limit(20))
display(df_quarantine.limit(20))

df_cleaned.write.mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('dev.silver.customer')

df_quarantine.write.mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('dev.silver.quarantine_customer')

print("✅ customer: silver + quarantine tables created")

In [0]:
%python
# ═══════════════════════════════════════════
# 5. PART — no foreign keys
#    PK: p_partkey
# ═══════════════════════════════════════════

# ── 1. Read from bronze with type casting & string cleaning ──
df = spark.read.table(f'{catalog_name}.bronze.part').select(
    F.when(
        F.col('p_partkey').cast('string') == '', None
    ).otherwise(F.col('p_partkey').cast('long')).alias('p_partkey'),

    F.nullif(F.trim(F.upper(F.col('p_name'))), F.lit('')).alias('p_name'),
    F.nullif(F.trim(F.upper(F.col('p_mfgr'))), F.lit('')).alias('p_mfgr'),
    F.nullif(F.trim(F.upper(F.col('p_brand'))), F.lit('')).alias('p_brand'),
    F.nullif(F.trim(F.upper(F.col('p_type'))), F.lit('')).alias('p_type'),
    F.when(
        F.col('p_size').cast('string') == '', None
    ).otherwise(F.col('p_size').cast('int')).alias('p_size'),
    F.nullif(F.trim(F.upper(F.col('p_container'))), F.lit('')).alias('p_container'),
    F.when(
        F.col('p_retailprice').cast('string') == '', None
    ).otherwise(F.col('p_retailprice').cast('decimal(18,2)')).alias('p_retailprice'),
    F.trim(F.col('p_comment')).alias('p_comment')
)

# ── 2. Validation ──
#     • INVALID_SIZE  : p_size <= 0
#     • INVALID_PRICE : p_retailprice <= 0
df_checked = (
    df.withColumn(
        'rnk',
        F.row_number().over(
            Window.partitionBy('p_partkey')
            .orderBy(F.desc('p_name'))
        )
    )
    .withColumn(
        'reason',
        F.concat_ws(
            ', ',
            F.when(F.col('rnk') > 1, F.lit('DUPLICATE_RECORD')),
            F.when(F.col('p_partkey').isNull(), F.lit('NULL_PK')),
            F.when(F.col('p_name').isNull(), F.lit('NULL_NAME')),
            F.when(
                F.col('p_size').isNull() | (F.col('p_size') <= 0),
                F.lit('INVALID_SIZE')
            ),
            F.when(
                F.col('p_retailprice').isNull() | (F.col('p_retailprice') <= 0),
                F.lit('INVALID_PRICE')
            )
        )
    )
)

# ── 3. Split clean vs quarantine ──
df_cleaned = (
    df_checked
    .filter(F.col('reason') == '')
    .drop('rnk', 'reason')
)

df_quarantine = (
    df_checked
    .filter(F.col('reason') != '')
    .drop('rnk')
)

# ── 4. Display & save ──
display(df_cleaned.limit(20))
display(df_quarantine.limit(20))

df_cleaned.write.mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{catalog_name}.silver.part')

df_quarantine.write.mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{catalog_name}.silver.quarantine_part')

print(f"✅ part: {catalog_name}.silver + quarantine tables created")

In [0]:
%python
# ═══════════════════════════════════════════
# 6. PARTSUPP — composite PK: (ps_partkey, ps_suppkey)
#    FK: ps_partkey → part.p_partkey
#    FK: ps_suppkey → supplier.s_suppkey
# ═══════════════════════════════════════════

# ── 1. Read from bronze with type casting & string cleaning ──
df = spark.read.table(f'{catalog_name}.bronze.partsupp').select(
    F.when(
        F.col('ps_partkey').cast('string') == '', None
    ).otherwise(F.col('ps_partkey').cast('long')).alias('ps_partkey'),

    F.when(
        F.col('ps_suppkey').cast('string') == '', None
    ).otherwise(F.col('ps_suppkey').cast('long')).alias('ps_suppkey'),

    F.when(
        F.col('ps_availqty').cast('string') == '', None
    ).otherwise(F.col('ps_availqty').cast('int')).alias('ps_availqty'),
    F.when(
        F.col('ps_supplycost').cast('string') == '', None
    ).otherwise(F.col('ps_supplycost').cast('decimal(18,2)')).alias('ps_supplycost'),
    F.trim(F.col('ps_comment')).alias('ps_comment')
)

# ── 2. Load FK reference tables ──
part_keys = (
    spark.read.table(f'{catalog_name}.silver.part')
    .select(F.col('p_partkey').alias('valid_partkey'))
    .distinct()
)

supplier_keys = (
    spark.read.table(f'{catalog_name}.silver.supplier')
    .select(F.col('s_suppkey').alias('valid_suppkey'))
    .distinct()
)

# ── 3. Validation ──
#     • NEGATIVE_QUANTITY : ps_availqty < 0
#     • NEGATIVE_COST     : ps_supplycost < 0
df_checked = (
    df.withColumn(
        'rnk',
        F.row_number().over(
            Window.partitionBy('ps_partkey', 'ps_suppkey')
            .orderBy(F.desc('ps_availqty'))
        )
    )
    .join(
        part_keys,
        F.col('ps_partkey') == F.col('valid_partkey'),
        'left'
    )
    .join(
        supplier_keys,
        F.col('ps_suppkey') == F.col('valid_suppkey'),
        'left'
    )
    .withColumn(
        'reason',
        F.concat_ws(
            ', ',
            F.when(F.col('rnk') > 1, F.lit('DUPLICATE_RECORD')),
            F.when(
                F.col('ps_partkey').isNull() | F.col('ps_suppkey').isNull(),
                F.lit('NULL_PK')
            ),
            F.when(
                F.col('ps_partkey').isNotNull() &
                F.col('valid_partkey').isNull(),
                F.lit('INVALID_FOREIGN_KEY_PART')
            ),
            F.when(
                F.col('ps_suppkey').isNotNull() &
                F.col('valid_suppkey').isNull(),
                F.lit('INVALID_FOREIGN_KEY_SUPPLIER')
            ),
            F.when(
                F.col('ps_availqty').isNotNull() & (F.col('ps_availqty') < 0),
                F.lit('NEGATIVE_QUANTITY')
            ),
            F.when(
                F.col('ps_supplycost').isNotNull() & (F.col('ps_supplycost') < 0),
                F.lit('NEGATIVE_COST')
            )
        )
    )
    .drop('valid_partkey', 'valid_suppkey')
)

# ── 4. Split clean vs quarantine ──
df_cleaned = (
    df_checked
    .filter(F.col('reason') == '')
    .drop('rnk', 'reason')
)

df_quarantine = (
    df_checked
    .filter(F.col('reason') != '')
    .drop('rnk')
)

# ── 5. Display & save ──
display(df_cleaned.limit(20))
display(df_quarantine.limit(20))

df_cleaned.write.mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{catalog_name}.silver.partsupp')

df_quarantine.write.mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{catalog_name}.silver.quarantine_partsupp')

print(f"✅ partsupp: {catalog_name}.silver + quarantine tables created")

In [0]:
%python
# ═══════════════════════════════════════════
# 7. ORDERS — FK: o_custkey → customer.c_custkey
#    PK: o_orderkey
#    Time window: o_orderdate between 1992-01-01 and 1998-12-31
# ═══════════════════════════════════════════

# ── 1. Read from bronze with type casting & string cleaning ──
df = spark.read.table(f'{catalog_name}.bronze.orders').select(
    F.when(
        F.col('o_orderkey').cast('string') == '', None
    ).otherwise(F.col('o_orderkey').cast('long')).alias('o_orderkey'),

    F.when(
        F.col('o_custkey').cast('string') == '', None
    ).otherwise(F.col('o_custkey').cast('long')).alias('o_custkey'),
    F.nullif(F.trim(F.upper(F.col('o_orderstatus'))), F.lit('')).alias('o_orderstatus'),
    F.when(
        F.col('o_totalprice').cast('string') == '', None
    ).otherwise(F.col('o_totalprice').cast('decimal(18,2)')).alias('o_totalprice'),
    F.col('o_orderdate').cast('date').alias('o_orderdate'),
    F.nullif(F.trim(F.upper(F.col('o_orderpriority'))), F.lit('')).alias('o_orderpriority'),
    F.nullif(F.trim(F.upper(F.col('o_clerk'))), F.lit('')).alias('o_clerk'),
    F.when(
        F.col('o_shippriority').cast('string') == '', None
    ).otherwise(F.col('o_shippriority').cast('int')).alias('o_shippriority'),
    F.trim(F.col('o_comment')).alias('o_comment')
)

# ── 2. Load FK reference table ──
customer_keys = (
    spark.read.table(f'{catalog_name}.silver.customer')
    .select(F.col('c_custkey').alias('valid_custkey'))
    .distinct()
)

# ── 3. Validation ──
#     • INVALID_STATUS    : o_orderstatus not in (O, F, P)
#     • INVALID_TOTALPRICE: o_totalprice <= 0
#     • DATE_OUT_OF_RANGE : o_orderdate not between 1992-01-01 and 1998-12-31
df_checked = (
    df.withColumn(
        'rnk',
        F.row_number().over(
            Window.partitionBy('o_orderkey')
            .orderBy(F.desc('o_custkey'))
        )
    )
    .join(
        customer_keys,
        F.col('o_custkey') == F.col('valid_custkey'),
        'left'
    )
    .withColumn(
        'reason',
        F.concat_ws(
            ', ',
            F.when(F.col('rnk') > 1, F.lit('DUPLICATE_RECORD')),
            F.when(F.col('o_orderkey').isNull(), F.lit('NULL_PK')),
            F.when(
                F.col('o_custkey').isNotNull() &
                F.col('valid_custkey').isNull(),
                F.lit('INVALID_FOREIGN_KEY')
            ),
            F.when(
                F.col('o_orderstatus').isNull() |
                ~F.col('o_orderstatus').isin(['O', 'F', 'P']),
                F.lit('INVALID_STATUS')
            ),
            F.when(
                F.col('o_totalprice').isNull() | (F.col('o_totalprice') <= 0),
                F.lit('INVALID_TOTALPRICE')
            ),
            F.when(
                F.col('o_orderdate').isNull() |
                (F.col('o_orderdate') < F.lit('1992-01-01').cast('date')) |
                (F.col('o_orderdate') > F.lit('1998-12-31').cast('date')),
                F.lit('DATE_OUT_OF_RANGE')
            )
        )
    )
    .drop('valid_custkey')
)

# ── 4. Split clean vs quarantine ──
df_cleaned = (
    df_checked
    .filter(F.col('reason') == '')
    .drop('rnk', 'reason')
)

df_quarantine = (
    df_checked
    .filter(F.col('reason') != '')
    .drop('rnk')
)

# ── 5. Display & save ──
display(df_cleaned.limit(20))
display(df_quarantine.limit(20))

df_cleaned.write.mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{catalog_name}.silver.orders')

df_quarantine.write.mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{catalog_name}.silver.quarantine_orders')

print(f"✅ orders: {catalog_name}.silver + quarantine tables created")

In [0]:
%python
# ═══════════════════════════════════════════
# 8. LINEITEM — composite PK: (l_orderkey, l_linenumber)
#    FK: l_orderkey → orders.o_orderkey
#    FK: l_partkey  → part.p_partkey
#    FK: l_suppkey  → supplier.s_suppkey
#    Time window: l_shipdate between 1992-01-01 and 1998-12-31
# ═══════════════════════════════════════════

# ── 1. Read from bronze with type casting & string cleaning ──
df = spark.read.table('dev.bronze.lineitem').select(
    F.when(
        F.col('l_orderkey').cast('string') == '', None
    ).otherwise(F.col('l_orderkey').cast('long')).alias('l_orderkey'),

    F.when(
        F.col('l_linenumber').cast('string') == '', None
    ).otherwise(F.col('l_linenumber').cast('int')).alias('l_linenumber'),

    F.when(
        F.col('l_partkey').cast('string') == '', None
    ).otherwise(F.col('l_partkey').cast('long')).alias('l_partkey'),
    F.when(
        F.col('l_suppkey').cast('string') == '', None
    ).otherwise(F.col('l_suppkey').cast('long')).alias('l_suppkey'),
    F.when(
        F.col('l_quantity').cast('string') == '', None
    ).otherwise(F.col('l_quantity').cast('decimal(18,2)')).alias('l_quantity'),
    F.when(
        F.col('l_extendedprice').cast('string') == '', None
    ).otherwise(F.col('l_extendedprice').cast('decimal(18,2)')).alias('l_extendedprice'),
    F.when(
        F.col('l_discount').cast('string') == '', None
    ).otherwise(F.col('l_discount').cast('decimal(18,2)')).alias('l_discount'),
    F.when(
        F.col('l_tax').cast('string') == '', None
    ).otherwise(F.col('l_tax').cast('decimal(18,2)')).alias('l_tax'),
    F.nullif(F.trim(F.upper(F.col('l_returnflag'))), F.lit('')).alias('l_returnflag'),
    F.nullif(F.trim(F.upper(F.col('l_linestatus'))), F.lit('')).alias('l_linestatus'),
    F.col('l_shipdate').cast('date').alias('l_shipdate'),
    F.col('l_commitdate').cast('date').alias('l_commitdate'),
    F.col('l_receiptdate').cast('date').alias('l_receiptdate'),
    F.nullif(F.trim(F.upper(F.col('l_shipinstruct'))), F.lit('')).alias('l_shipinstruct'),
    F.nullif(F.trim(F.upper(F.col('l_shipmode'))), F.lit('')).alias('l_shipmode'),
    F.trim(F.col('l_comment')).alias('l_comment')
)

# ── 2. Load FK reference tables ──
orders_keys = (
    spark.read.table('dev.silver.orders')
    .select(F.col('o_orderkey').alias('valid_orderkey'))
    .distinct()
)

part_keys = (
    spark.read.table('dev.silver.part')
    .select(F.col('p_partkey').alias('valid_partkey'))
    .distinct()
)

supplier_keys = (
    spark.read.table('dev.silver.supplier')
    .select(F.col('s_suppkey').alias('valid_suppkey'))
    .distinct()
)

# ── 3. Validation ──
#     • INVALID_QUANTITY     : l_quantity <= 0
#     • INVALID_EXTENDEDPRICE: l_extendedprice <= 0
#     • INVALID_DISCOUNT     : l_discount < 0 or > 0.10
#     • INVALID_TAX          : l_tax < 0 or > 0.08
#     • INVALID_RETURNFLAG   : l_returnflag not in (A, N, R)
#     • INVALID_LINESTATUS   : l_linestatus not in (O, F)
#     • DATE_OUT_OF_RANGE    : l_shipdate not between 1992-01-01 and 1998-12-31
df_checked = (
    df.withColumn(
        'rnk',
        F.row_number().over(
            Window.partitionBy('l_orderkey', 'l_linenumber')
            .orderBy(F.desc('l_partkey'))
        )
    )
    .join(orders_keys,   F.col('l_orderkey') == F.col('valid_orderkey'), 'left')
    .join(part_keys,     F.col('l_partkey')  == F.col('valid_partkey'),  'left')
    .join(supplier_keys, F.col('l_suppkey')  == F.col('valid_suppkey'),  'left')
    .withColumn(
        'reason',
        F.concat_ws(
            ', ',
            F.when(F.col('rnk') > 1, F.lit('DUPLICATE_RECORD')),
            F.when(
                F.col('l_orderkey').isNull() | F.col('l_linenumber').isNull(),
                F.lit('NULL_PK')
            ),
            F.when(
                F.col('l_orderkey').isNotNull() &
                F.col('valid_orderkey').isNull(),
                F.lit('INVALID_FOREIGN_KEY_ORDER')
            ),
            F.when(
                F.col('l_partkey').isNotNull() &
                F.col('valid_partkey').isNull(),
                F.lit('INVALID_FOREIGN_KEY_PART')
            ),
            F.when(
                F.col('l_suppkey').isNotNull() &
                F.col('valid_suppkey').isNull(),
                F.lit('INVALID_FOREIGN_KEY_SUPPLIER')
            ),
            F.when(
                F.col('l_quantity').isNull() | (F.col('l_quantity') <= 0),
                F.lit('INVALID_QUANTITY')
            ),
            F.when(
                F.col('l_extendedprice').isNull() | (F.col('l_extendedprice') <= 0),
                F.lit('INVALID_EXTENDEDPRICE')
            ),
            F.when(
                F.col('l_discount').isNull() |
                (F.col('l_discount') < 0) |
                (F.col('l_discount') > 0.10),
                F.lit('INVALID_DISCOUNT')
            ),
            F.when(
                F.col('l_tax').isNull() |
                (F.col('l_tax') < 0) |
                (F.col('l_tax') > 0.08),
                F.lit('INVALID_TAX')
            ),
            F.when(
                F.col('l_returnflag').isNull() |
                ~F.col('l_returnflag').isin(['A', 'N', 'R']),
                F.lit('INVALID_RETURNFLAG')
            ),
            F.when(
                F.col('l_linestatus').isNull() |
                ~F.col('l_linestatus').isin(['O', 'F']),
                F.lit('INVALID_LINESTATUS')
            ),
            F.when(
                F.col('l_shipdate').isNull() |
                (F.col('l_shipdate') < F.lit('1992-01-01').cast('date')) |
                (F.col('l_shipdate') > F.lit('1998-12-31').cast('date'))|
                (F.col('l_shipdate') > F.col('l_receiptdate')),
                F.lit('DATE_OUT_OF_RANGE')
            )
        )
    )
    .drop('valid_orderkey', 'valid_partkey', 'valid_suppkey')
)

# ── 4. Split clean vs quarantine ──
df_cleaned = (
    df_checked
    .filter(F.col('reason') == '')
    .drop('rnk', 'reason')
)

df_quarantine = (
    df_checked
    .filter(F.col('reason') != '')
    .drop('rnk')
)

# ── 5. Display & save ──
display(df_cleaned.limit(20))
display(df_quarantine.limit(20))

df_cleaned.write.mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('dev.silver.lineitem')

df_quarantine.write.mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('dev.silver.quarantine_lineitem')

print("✅ lineitem: silver + quarantine tables created")

In [0]:
%python
dbutils.widgets.text("catalog_name" ,"dev")
catalog_name = dbutils.widgets.get("catalog_name")

In [0]:
%python
# ═══════════════════════════════════════════
# SUMMARY — row counts for all silver + quarantine tables
# ═══════════════════════════════════════════

tables = [
    'region', 'nation', 'supplier', 'customer',
    'part', 'partsupp', 'orders', 'lineitem'
]

rows = []
for t in tables:
    bronze_cnt = spark.read.table(f'dev.bronze.{t}').count()
    clean_cnt  = spark.read.table(f'dev.silver.{t}').count()
    quar_cnt   = spark.read.table(f'dev.silver.quarantine_{t}').count()
    total_cnt  = clean_cnt + quar_cnt
    status     = 'PASS' if total_cnt == bronze_cnt else 'FAIL'
    rows.append((t, bronze_cnt, clean_cnt, quar_cnt, total_cnt, status))

summary_df = spark.createDataFrame(
    rows,
    ['table', 'bronze_rows', 'clean_rows', 'quarantine_rows', 'silver_plus_quarantine', 'status']
)
display(summary_df)

summary_df.write.mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{catalog_name}.ops.row_counts_reconcilation')

print("✅ Reconciliation summary saved to dev.ops.row_counts_reconcilation")

failed = summary_df.filter(F.col('status') == 'FAIL')
if failed.count() > 0:
    print('❌ Row count mismatch (silver + quarantine != bronze) for:')
    failed.display()
else:
    print('✅ All tables passed: silver + quarantine == bronze')